# Trabalho Prático 1 - Estudo Comparativo de Modelos (VERSÃO COM TODOS OS MODELOS)
## Previsão de Preços de Carros Usados

**Licenciatura em Engenharia de Sistemas e Tecnologias Informáticas** 

**Unidade Curricular:** Aprendizagem Automática

---

### 1. Introdução e Objetivos
Este trabalho visa o desenvolvimento e comparação de múltiplos modelos de regressão para a previsão de preços de automóveis. O objetivo é identificar qual a família de algoritmos que melhor generaliza para este domínio específico, evitando problemas comuns de *overfitting* (sobreajuste) e *underfitting* (subajuste).

### 2. Modelos em Estudo
Conforme os requisitos do projeto, serão implementados e avaliados os seguintes algoritmos:
1.  **Regressão Linear (Ridge):** Modelo base linear com regularização L2.
2.  **K-Nearest Neighbors (KNN):** Algoritmo baseado em distância.
3.  **Árvores de Decisão:** Modelo base não-linear.
4.  **Random Forest & Variantes (XGBoost):** Métodos de *Ensemble* (Bagging e Boosting).
5.  **Support Vector Machines (SVM):** Regressão com vetores de suporte (LinearSVR).
6.  **Redes Neuronais (MLP):** Perceptron Multicamada para capturar relações complexas.

### 3. Metodologia de Prevenção de Overfitting
Para garantir a robustez dos resultados:
* Utilização de **Validação Cruzada (K-Fold)** em todos os treinos.
* Aplicação de **Regularização** (L1/L2, profundidade máxima, número mínimo de amostras por folha).
* **Imputação Hierárquica** de dados para garantir a integridade do *dataset*.
* **Early Stopping** nas Redes Neuronais e Boosting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os, joblib, re, random

# Processamento
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# --- MODELOS SOLICITADOS ---
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.svm import LinearSVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

# Configuração Global
MODE = 'full'  # Usar 'full' para treinar todos os modelos corretamente
TRAIN_N_JOBS = -1
RANDOM_STATE = 42

if MODE == 'quick':
    CV_FOLDS = 3
    N_ITER = 2
else:
    CV_FOLDS = 5
    N_ITER = 10 

os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Bibliotecas carregadas e modelos importados.")

### 4. Tratamento e Engenharia de Dados
Utiliza-se a estratégia de **Imputação Hierárquica** (Marca+Modelo -> Marca -> Global) que demonstrou reduzir significativamente o erro nas iterações anteriores. As variáveis categóricas são tratadas com `LabelEncoder` para manter a dimensionalidade controlada, essencial para modelos como SVM e KNN.

In [ ]:
# Carregar Dados
try:
    train_df = pd.read_csv('../data/train.csv')
    test_df = pd.read_csv('../data/test.csv')
except:
    print("Erro ao carregar ficheiros.")

# --- 1. LIMPEZA E EXTRAÇÃO ---
def clean_engine(row):
    engine = str(row['engine'])
    hp, liters, cylinders = np.nan, np.nan, np.nan
    
    hp_match = re.search(r'(\d+\.?\d*)HP', engine)
    if hp_match: hp = float(hp_match.group(1))
    
    lit_match = re.search(r'(\d+\.?\d*)L', engine)
    if not lit_match: lit_match = re.search(r'(\d+\.?\d*) Liter', engine)
    if lit_match: liters = float(lit_match.group(1))
    
    cyl_match = re.search(r'V(\d+)', engine)
    if not cyl_match: cyl_match = re.search(r'(\d+) Cylinder', engine)
    if cyl_match: cylinders = float(cyl_match.group(1))
    return pd.Series([hp, liters, cylinders])

def clean_initial(df):
    data = df.copy()
    # Normalizar Mileage
    if 'mileage' in data.columns:
        if data['mileage'].dtype == 'O':
             data['milage'] = data['mileage'].astype(str).str.replace(',', '').str.extract(r'(\d+)')[0].astype(float)
        else:
             data['milage'] = data['mileage']
        if 'milage' != 'mileage':
            data.drop(columns=['mileage'], inplace=True, errors='ignore')
    elif 'milage' in data.columns:
        if data['milage'].dtype == 'O':
            data['milage'] = data['milage'].astype(str).str.replace(',', '').str.extract(r'(\d+)')[0].astype(float)
    # Extração
    data[['HP', 'Liters', 'Cylinders']] = data.apply(clean_engine, axis=1)
    data['age'] = 2025 - pd.to_numeric(data['model_year'], errors='coerce')
    return data

train_df = clean_initial(train_df)
test_df = clean_initial(test_df)

# --- 2. IMPUTAÇÃO HIERÁRQUICA ---
train_df['brand_model'] = train_df['brand'].astype(str) + "_" + train_df['model'].astype(str)
test_df['brand_model'] = test_df['brand'].astype(str) + "_" + test_df['model'].astype(str)

map_bm_hp = train_df.groupby('brand_model')['HP'].median()
map_b_hp = train_df.groupby('brand')['HP'].median()
global_hp = train_df['HP'].median()

map_bm_lit = train_df.groupby('brand_model')['Liters'].median()
map_b_lit = train_df.groupby('brand')['Liters'].median()
global_lit = train_df['Liters'].median()

def hierarchical_fill(row, col, map_bm, map_b, glob):
    if pd.isna(row[col]):
        val = map_bm.get(row['brand_model'])
        if pd.notna(val): return val
        val = map_b.get(row['brand'])
        if pd.notna(val): return val
        return glob
    return row[col]

for df in [train_df, test_df]:
    df['HP'] = df.apply(lambda x: hierarchical_fill(x, 'HP', map_bm_hp, map_b_hp, global_hp), axis=1)
    df['Liters'] = df.apply(lambda x: hierarchical_fill(x, 'Liters', map_bm_lit, map_b_lit, global_lit), axis=1)
    for c in ['Cylinders', 'milage']:
        df[c] = df[c].fillna(train_df[c].median())
    
    df['log_milage'] = np.log1p(df['milage'])
    df['hp_per_liter'] = df['HP'] / df['Liters'].replace(0, 1.6)

# --- 3. LABEL ENCODING ---
cat_cols = ['brand', 'model', 'fuel_type', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title']
for col in cat_cols:
    le = LabelEncoder()
    full_data = pd.concat([train_df[col].astype(str), test_df[col].astype(str)], axis=0)
    le.fit(full_data)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))

print("Engenharia de dados completa.")

In [ ]:
features = ['brand', 'model', 'age', 'log_milage', 'HP', 'Liters', 'Cylinders', 'hp_per_liter',
            'fuel_type', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title']

X = train_df[features]
y = np.log1p(train_df['price'])

# Pipeline de Pré-processamento
# RobustScaler é essencial para SVM, KNN e Redes Neuronais (sensíveis à escala)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), features)
    ])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Dados divididos em Treino/Validação.")

### 5. Configuração e Treino dos Modelos
Abaixo define-se a grelha de treino para todos os modelos solicitados. Utilizamos `Pipeline` para evitar fugas de informação (*data leakage*) e `RandomizedSearchCV` para otimização eficiente.

In [ ]:
models_list = []

# 1. Regressão Linear (Ridge - Regularização L2)
models_list.append(('Linear (Ridge)', 
                    Pipeline([('pre', preprocessor), ('model', Ridge())]),
                    {'model__alpha': [1.0, 10.0, 50.0]}))

# 2. KNN (K-Nearest Neighbors)
models_list.append(('KNN', 
                    Pipeline([('pre', preprocessor), ('model', KNeighborsRegressor(n_jobs=TRAIN_N_JOBS))]),
                    {'model__n_neighbors': [5, 10, 20], 'model__weights': ['uniform', 'distance']}))

# 3. Árvore de Decisão
models_list.append(('Decision Tree', 
                    Pipeline([('pre', preprocessor), ('model', DecisionTreeRegressor(random_state=RANDOM_STATE))]),
                    {'model__max_depth': [10, 20, 30], 'model__min_samples_leaf': [5, 10]}))

# 4. Random Forest
models_list.append(('Random Forest', 
                    Pipeline([('pre', preprocessor), ('model', RandomForestRegressor(n_jobs=TRAIN_N_JOBS, random_state=RANDOM_STATE))]),
                    {'model__n_estimators': [200, 300], 'model__max_depth': [15, 20], 'model__min_samples_leaf': [2, 4]}))

# 5. XGBoost (Variante de RF - Gradient Boosting)
models_list.append(('XGBoost', 
                    Pipeline([('pre', preprocessor), ('model', XGBRegressor(n_jobs=TRAIN_N_JOBS, random_state=RANDOM_STATE))]),
                    {'model__n_estimators': [1000, 3000], 'model__learning_rate': [0.01, 0.05], 
                     'model__max_depth': [6, 8], 'model__reg_alpha': [0.1], 'model__reg_lambda': [1.0]}))

# 6. SVM (LinearSVR - Escalável para grandes datasets)
# SVR padrão é O(n^3) e não terminaria em tempo útil para este dataset.
models_list.append(('SVM (Linear)', 
                    Pipeline([('pre', preprocessor), ('model', LinearSVR(random_state=RANDOM_STATE, max_iter=2000))]),
                    {'model__C': [0.1, 1.0, 10.0]}))

# 7. Redes Neuronais (MLP)
models_list.append(('Neural Net (MLP)', 
                    Pipeline([('pre', preprocessor), ('model', MLPRegressor(random_state=RANDOM_STATE, early_stopping=True))]),
                    {'model__hidden_layer_sizes': [(100,), (100, 50)], 'model__alpha': [0.0001, 0.001]}))

# --- LOOP DE TREINO ---
results = []
trained_estimators = []
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print("Iniciando treino de todos os modelos...")

for name, pipeline, params in models_list:
    print(f"\n>>> {name}...")
    search = RandomizedSearchCV(pipeline, params, n_iter=N_ITER, cv=cv, 
                                scoring='neg_mean_squared_error', n_jobs=TRAIN_N_JOBS, 
                                random_state=RANDOM_STATE)
    try:
        search.fit(X_train, y_train)
        best = search.best_estimator_
        
        # Avaliação
        preds = best.predict(X_val)
        rmse = np.sqrt(mean_squared_error(np.expm1(y_val), np.clip(np.expm1(preds), 0, None)))
        r2 = r2_score(y_val, preds)
        
        print(f"    RMSE: {rmse:,.0f} | R2: {r2:.4f}")
        results.append({'Modelo': name, 'RMSE': rmse, 'R2': r2})
        trained_estimators.append((name, best))
        
        # Salvar individualmente
        safe_name = name.replace(" ", "_").replace("(", "").replace(")", "")
        try: joblib.dump(best, f'../models/{safe_name}_model.joblib')
        except: pass
        
    except Exception as e:
        print(f"    Erro ao treinar {name}: {e}")

df_results = pd.DataFrame(results).sort_values('RMSE')
print("\n--- RESULTADOS FINAIS ---")
display(df_results)

### 6. Ensemble Final e Submissão
Com base nos resultados, construímos um *Voting Regressor* utilizando os melhores modelos (tipicamente XGBoost e Random Forest) para gerar a submissão final.

In [ ]:
# Selecionar os 2 melhores modelos automaticamente
top_2_models = df_results.head(2)['Modelo'].values
estimators_for_voting = [(name, model) for name, model in trained_estimators if name in top_2_models]

print(f"\nCriando Ensemble com: {top_2_models}")

if len(estimators_for_voting) >= 1:
    # Pesos manuais se XGB estiver presente (damos-lhe prioridade)
    weights = [0.7, 0.3] if 'XGBoost' in top_2_models else None
    
    ensemble = VotingRegressor(estimators=estimators_for_voting, weights=weights, n_jobs=TRAIN_N_JOBS)
    ensemble.fit(X_train, y_train)
    
    # Avaliar
    ens_preds = ensemble.predict(X_val)
    ens_rmse = np.sqrt(mean_squared_error(np.expm1(y_val), np.clip(np.expm1(ens_preds), 0, None)))
    print(f"RMSE Ensemble: {ens_rmse:,.2f}")
    
    # Submissão
    X_test_sub = test_df[features]
    final_log = ensemble.predict(X_test_sub)
    final_real = np.expm1(final_log)
    
    sub = pd.DataFrame({'id': test_df['id'], 'price': np.clip(final_real, 0, None)})
    sub.to_csv('submission_all_models.csv', index=False)
    print("Ficheiro 'submission_all_models.csv' gerado.")
else:
    print("Erro: Não há modelos suficientes para o ensemble.")

In [ ]:
# Visualização Comparativa
plt.figure(figsize=(10, 5))
sns.barplot(data=df_results, x='RMSE', y='Modelo', palette='viridis')
plt.title('Comparação de RMSE (Menor é melhor)')
plt.xlabel('RMSE ($)')
plt.show()